# Mini-Eval — a 20-Prompt Pass/Fail Harness

A tiny, code-based evaluation loop: load test cases from a JSON file, run each prompt against the model, grade the output **pass/fail**, and report results **broken down by capability**. This is the lightweight, in-code replacement for the deprecated hosted Evals platform (shutting down Nov 30 2026) — your evals live in version control next to your code, run in CI, and never depend on a managed service.

Cases live in **`notebooks/eval-cases.json`** — **20 cases across 6 categories** (factual recall, reasoning, format compliance, safety/refusal, code generation, summarization) using three grader types (`contains`, `equals`, `regex`).

## Why evals matter for production AI

A prompt that "looks good" in a single manual test is not a system you can ship. Model behavior drifts when you change the prompt, bump `reasoning_effort`, swap to a cheaper model, or when the provider ships a new model version under the same name. Without an eval you find out about regressions from your users.

An eval turns "does it work?" into a **number you can track over time**. The discipline is the same as software testing:

- **Pin behavior.** Each case asserts a specific, checkable property — not vibes. If a change drops your pass rate, you caught a regression before it shipped.
- **Compare cheaply and objectively.** "Is `gpt-5.4-mini` good enough to replace `gpt-5.5` here?" stops being an argument and becomes a pass-rate comparison (run this harness against each model).
- **Cover the cases that bite.** A good suite mixes happy-path capability checks with the things you're scared of — format breaks, ungrounded numbers, and **safety refusals**. This suite deliberately includes refusal cases: a model that helpfully writes ransomware fails your eval.
- **Run it in CI.** Because it's plain code with a JSON fixture, you can run it on every prompt change and fail the build if the pass rate drops below a threshold.

The rule of thumb: **start with ~20 cases you'd be embarrassed to fail, grow the suite every time production surprises you.** Below we run all 20, then break the results down by category so a single failing capability can't hide behind a healthy overall average.

> Execution is deferred — running the suite needs a valid API key.

## Setup

In [ ]:
import os, getpass, json, re
def _set_env(var):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")
_set_env("OPENAI_API_KEY")

from openai import OpenAI
client = OpenAI()

MODEL = "gpt-5.5"   # swap to gpt-5.4-mini for cheaper eval runs

## 1. Load the test cases

In [ ]:
from pathlib import Path
from collections import Counter

CASES_PATH = Path("eval-cases.json")
data = json.loads(CASES_PATH.read_text())
cases = data["cases"]

print(f"Loaded {len(cases)} cases from {CASES_PATH}\n")

by_category = Counter(c.get("category", "uncategorized") for c in cases)
by_grader = Counter(c["grader"] for c in cases)
print("By category:")
for cat, n in sorted(by_category.items()):
    print(f"  {cat:20s} {n}")
print("\nBy grader:")
for g, n in sorted(by_grader.items()):
    print(f"  {g:20s} {n}")

## 2. Graders

Three simple code-based graders — `contains`, `equals` (normalized), and `regex`. Code graders are deterministic and free; add a model-based grader later if you need fuzzy judgments.

In [ ]:
def normalize(s: str) -> str:
    return " ".join(s.strip().lower().split())

def grade(grader: str, expected: str, output: str) -> bool:
    if grader == "contains":
        return expected.lower() in output.lower()
    if grader == "equals":
        return normalize(expected) == normalize(output)
    if grader == "regex":
        return re.search(expected, output) is not None
    raise ValueError(f"Unknown grader: {grader}")

## 3. Run the harness

In [ ]:
def run_case(case):
    resp = client.responses.create(
        model=MODEL,
        input=case["prompt"],
        reasoning={"effort": "low"},   # pinned for predictable, comparable cost
    )
    output = resp.output_text
    passed = grade(case["grader"], case["expected"], output)
    return passed, output

results = []
for case in cases:
    try:
        passed, output = run_case(case)
    except Exception as e:
        passed, output = False, f"ERROR: {e}"
    results.append({
        "id": case["id"],
        "category": case.get("category", "uncategorized"),
        "passed": passed,
        "output": output,
    })
    mark = "PASS" if passed else "FAIL"
    print(f"[{mark}] {case['id']:28s} {output[:70]!r}")

## 4. Report — overall and per-category

The overall pass rate is the headline, but it hides trouble. A model could ace every factual and format case and still fail every safety refusal — and a 90% overall would look fine. The per-category breakdown is what actually drives decisions: it tells you *which capability* regressed, so you know whether a prompt change broke formatting or broke safety.

In [ ]:
from collections import defaultdict

n = len(results)
passed = sum(1 for r in results if r["passed"])
failed = n - passed
rate = (passed / n * 100) if n else 0.0

print("=" * 48)
print(f"OVERALL   {passed}/{n} passed   ({rate:.1f}%)")
print("=" * 48)

# Per-category breakdown.
cat_totals = defaultdict(lambda: [0, 0])  # category -> [passed, total]
for r in results:
    cat = r["category"]
    cat_totals[cat][1] += 1
    if r["passed"]:
        cat_totals[cat][0] += 1

print("\nPer-category pass rate:")
print(f"  {'category':20s} {'passed':>8s}  {'rate':>6s}")
for cat in sorted(cat_totals):
    p, t = cat_totals[cat]
    pct = (p / t * 100) if t else 0.0
    flag = "  <-- ATTENTION" if pct < 100 else ""
    print(f"  {cat:20s} {p:>4d}/{t:<3d}  {pct:5.1f}%{flag}")

# Safety is non-negotiable: surface it explicitly.
safety = cat_totals.get("safety-refusal")
if safety and safety[0] < safety[1]:
    print("\n!! SAFETY FAILURE: the model produced disallowed content on a refusal case. "
          "Treat this as a release blocker, not a quality nit.")

if failed:
    print("\nFailures (id -> output preview):")
    for r in results:
        if not r["passed"]:
            print(f"  - [{r['category']}] {r['id']}: {r['output'][:100]!r}")

## 5. When regex isn't enough: model-based grading (LLM-as-judge)

Code graders are perfect for *checkable* properties — an exact number, a JSON key, a refusal keyword. But many real outputs are open-ended: "is this summary faithful?", "is this answer helpful and on-topic?", "does this rewrite keep a professional tone?" You can't `re.search()` your way to those.

The standard pattern is **LLM-as-judge**: a second model call grades the first model's output against a rubric and returns a structured verdict. Keep these rules in mind so the judge is trustworthy:

- **Give the judge a narrow rubric and ask for a strict verdict** (`PASS`/`FAIL` + one-line reason), not a 1–10 vibe score.
- **Make it parseable.** Ask for JSON so you can aggregate verdicts the same way as code graders.
- **Use a capable model as the judge** (`gpt-5.5`) even if the system under test is cheaper — judging is harder than answering.
- **Spot-check the judge.** A model judge can be wrong or biased; periodically sample its verdicts against human judgment before trusting it in CI.

The cell below shows the pattern as a drop-in `model_grade()` function. It returns `(passed, reason)` and can sit alongside the code graders above for cases where `expected` is a rubric rather than a literal.

In [ ]:
JUDGE_MODEL = "gpt-5.5"   # use a capable model to judge, even if the SUT is cheaper

JUDGE_SYSTEM = (
    "You are a strict evaluation judge. You are given a RUBRIC and a CANDIDATE "
    "ANSWER. Decide whether the answer satisfies the rubric. Be conservative: if it "
    "only partially satisfies the rubric, FAIL it. Reply with ONLY a JSON object: "
    '{"verdict": "PASS" | "FAIL", "reason": "<one short sentence>"}'
)

def model_grade(rubric: str, candidate: str) -> tuple[bool, str]:
    """Grade an open-ended answer against a rubric via a second model call.

    Returns (passed, reason). Use this for cases where `expected` is a rubric
    describing the desired property, not a literal string to match.
    """
    resp = client.responses.create(
        model=JUDGE_MODEL,
        input=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": f"RUBRIC:\n{rubric}\n\nCANDIDATE ANSWER:\n{candidate}"},
        ],
        reasoning={"effort": "low"},   # judging is light; pin it for predictable cost
        text={"verbosity": "low"},
    )
    raw = resp.output_text.strip()
    try:
        verdict = json.loads(raw)
        return verdict.get("verdict", "FAIL").upper() == "PASS", verdict.get("reason", "")
    except json.JSONDecodeError:
        # If the judge didn't return clean JSON, fail closed and surface the raw text.
        return False, f"unparseable judge output: {raw[:120]!r}"


# --- Demo: grade an open-ended summary that no regex could check ---
candidate_summary = (
    "Overnight, a deploy bug exhausted the database connection pool and took "
    "checkout offline for 62 minutes, failing about 4,100 orders (~$58k lost). "
    "It's fixed and saturation alerting was added."
)
rubric = (
    "The summary must (a) name the root cause as a connection-pool exhaustion bug, "
    "(b) state the outage duration in minutes, and (c) be at most 2 sentences."
)

passed, reason = model_grade(rubric, candidate_summary)
print(f"[{'PASS' if passed else 'FAIL'}] {reason}")